In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import numpy as np

from src.utils import (
    get_args,
    set_seed,
    get_datesets_and_loaders,
    get_trained_VAE,
    get_trained_VAE_with_domain_classifier,
    get_trained_classifier,
    get_trained_classifier_Base,
    test_model,
    prepare_report,
    run_all_senario
)

/home/asad/workspace/DomainProject/changeDomain/notebooks/effective-gzsda/gzsda/src/utils.py:3: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.1)
  import scipy


In [3]:
DOMAIN_SET =['regu','xray']
DATA_DIR = './data/XrayBaggage20/'
DATASET_DETAILS = {
    "prefix": 'XrayDataset-',
    "suffix": '-resnet101-noft.mat',
    "resnet_feature": 'resnet101_features',
    "split_file_name": 'instanceSplit_xrayDataset_unseen10.mat',
}
NUM_LABELS=20

In [4]:
result = {}

## Base

In [5]:
def main_base(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    classifier = get_trained_classifier_Base(
        data_loaders=data_loaders,
        NUM_LABELS=NUM_LABELS,
        device=device)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [6]:
result["Base"] = run_all_senario(main_base, DOMAIN_SET)

regu -> xray
per-class acc:0.4053, seen acc:0.7863, unseen acc:0.0243, H:0.0471
per-class acc:0.4153, seen acc:0.8006, unseen acc:0.0300, H:0.0578
per-class acc:0.4313, seen acc:0.8399, unseen acc:0.0227, H:0.0442
per-class acc:0.4687, seen acc:0.8970, unseen acc:0.0404, H:0.0773
per-class acc:0.4485, seen acc:0.8865, unseen acc:0.0105, H:0.0208
Seen:     84.21 ± 2.22
Unseen:   2.56 ± 0.49
H-mean:   4.94 ± 0.92
xray -> regu
per-class acc:0.5703, seen acc:0.9147, unseen acc:0.2260, H:0.3624
per-class acc:0.5080, seen acc:0.9502, unseen acc:0.0657, H:0.1230
per-class acc:0.6180, seen acc:0.9577, unseen acc:0.2783, H:0.4312
per-class acc:0.6343, seen acc:0.9736, unseen acc:0.2951, H:0.4529
per-class acc:0.5462, seen acc:0.9368, unseen acc:0.1556, H:0.2669
Seen:     94.66 ± 1.00
Unseen:   20.41 ± 4.23
H-mean:   32.73 ± 6.05


# GZSDA

In [7]:
def main_gzsda(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [8]:
result["GZSDA"] = run_all_senario(main_gzsda, DOMAIN_SET)

regu -> xray
per-class acc:0.4656, seen acc:0.7348, unseen acc:0.1963, H:0.3099
per-class acc:0.4984, seen acc:0.7354, unseen acc:0.2613, H:0.3856
per-class acc:0.5644, seen acc:0.7746, unseen acc:0.3543, H:0.4862
per-class acc:0.5979, seen acc:0.8503, unseen acc:0.3454, H:0.4913
per-class acc:0.5505, seen acc:0.7830, unseen acc:0.3179, H:0.4522
Seen:     77.56 ± 2.11
Unseen:   29.50 ± 2.95
H-mean:   42.50 ± 3.44
xray -> regu
per-class acc:0.7299, seen acc:0.8841, unseen acc:0.5757, H:0.6973
per-class acc:0.6659, seen acc:0.9076, unseen acc:0.4242, H:0.5782
per-class acc:0.7453, seen acc:0.9106, unseen acc:0.5799, H:0.7086
per-class acc:0.7194, seen acc:0.9416, unseen acc:0.4972, H:0.6508
per-class acc:0.7065, seen acc:0.8896, unseen acc:0.5235, H:0.6591
Seen:     90.67 ± 1.01
Unseen:   52.01 ± 2.86
H-mean:   65.88 ± 2.29


## m0

In [9]:
def main_m0(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        change_policy_epoch=30)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [10]:
result["Our"] = run_all_senario(main_m0, DOMAIN_SET)

regu -> xray
per-class acc:0.4640, seen acc:0.6980, unseen acc:0.2300, H:0.3460
per-class acc:0.5138, seen acc:0.7058, unseen acc:0.3217, H:0.4419
per-class acc:0.5710, seen acc:0.7345, unseen acc:0.4075, H:0.5242
per-class acc:0.5987, seen acc:0.8266, unseen acc:0.3709, H:0.5120
per-class acc:0.5533, seen acc:0.7444, unseen acc:0.3621, H:0.4872
Seen:     74.19 ± 2.29
Unseen:   33.84 ± 3.04
H-mean:   46.23 ± 3.23
xray -> regu
per-class acc:0.7279, seen acc:0.8467, unseen acc:0.6091, H:0.7085
per-class acc:0.6668, seen acc:0.8666, unseen acc:0.4670, H:0.6070
per-class acc:0.7350, seen acc:0.8750, unseen acc:0.5950, H:0.7084
per-class acc:0.7050, seen acc:0.9100, unseen acc:0.5000, H:0.6454
per-class acc:0.7150, seen acc:0.8691, unseen acc:0.5609, H:0.6818
Seen:     87.35 ± 1.03
Unseen:   54.64 ± 2.74
H-mean:   67.02 ± 1.96


## m1: seperate after encoder

In [11]:
def main_m1(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE_with_domain_classifier(
        data_loaders=data_loaders,
        args=args,
        device=device)
        
    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        change_policy_epoch=30)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [12]:
result["Our+GRE"] = run_all_senario(main_m1, DOMAIN_SET)

regu -> xray
per-class acc:0.4829, seen acc:0.6792, unseen acc:0.2867, H:0.4032
per-class acc:0.4877, seen acc:0.6736, unseen acc:0.3018, H:0.4169
per-class acc:0.5412, seen acc:0.7291, unseen acc:0.3534, H:0.4760
per-class acc:0.5717, seen acc:0.8100, unseen acc:0.3334, H:0.4724
per-class acc:0.5436, seen acc:0.7237, unseen acc:0.3634, H:0.4839
Seen:     72.31 ± 2.45
Unseen:   32.77 ± 1.47
H-mean:   45.05 ± 1.68
xray -> regu
per-class acc:0.7263, seen acc:0.8519, unseen acc:0.6008, H:0.7046
per-class acc:0.6812, seen acc:0.8667, unseen acc:0.4957, H:0.6307
per-class acc:0.7384, seen acc:0.8748, unseen acc:0.6020, H:0.7132
per-class acc:0.7054, seen acc:0.8938, unseen acc:0.5170, H:0.6551
per-class acc:0.7011, seen acc:0.8561, unseen acc:0.5461, H:0.6669
Seen:     86.87 ± 0.75
Unseen:   55.23 ± 2.16
H-mean:   67.41 ± 1.54



## Merge results

In [13]:
import pandas as pd
import re

rows = [(k, m, result[m][k]) for m in result for k in result[m]]
df = pd.DataFrame(rows, columns=['domain', 'method', 'values'])

def extract_metrics(text):
    matches = dict(re.findall(r'(\w+):\s+([\d.]+\s*±\s*[\d.]+)', text))
    return pd.Series(matches)

df[['seen', 'unseen', 'H-mean']] = df['values'].apply(extract_metrics)
df = df[['domain', 'method', 'seen', 'unseen', 'H-mean']]

df['method'] = pd.Categorical(df['method'], categories=['Base', 'GZSDA', 'Our', 'Our+GRE'], ordered=True)
df = df.sort_values(['domain', 'method']).reset_index(drop=True)

df

,domain,method,seen,unseen,H-mean
0,regu -> xray,Base,84.21 ± 2.22,2.56 ± 0.49,4.94 ± 0.92
1,regu -> xray,GZSDA,77.56 ± 2.11,29.50 ± 2.95,42.50 ± 3.44
2,regu -> xray,Our,74.19 ± 2.29,33.84 ± 3.04,46.23 ± 3.23
3,regu -> xray,Our+GRE,72.31 ± 2.45,32.77 ± 1.47,45.05 ± 1.68
4,xray -> regu,Base,94.66 ± 1.00,20.41 ± 4.23,32.73 ± 6.05
5,xray -> regu,GZSDA,90.67 ± 1.01,52.01 ± 2.86,65.88 ± 2.29
6,xray -> regu,Our,87.35 ± 1.03,54.64 ± 2.74,67.02 ± 1.96
7,xray -> regu,Our+GRE,86.87 ± 0.75,55.23 ± 2.16,67.41 ± 1.54


In [14]:
df.to_csv("./result_Xray.csv")